# Hotel Reviews - Text Preprocessing for Topic Modeling

This notebook preprocesses hotel review text for topic modeling analysis.

**Goals:**
1. Load and filter hotel reviews dataset
2. Remove placeholder reviews ("No Negative", "No Positive")
3. Clean and tokenize text
4. Create separate datasets for negative and positive reviews
5. Export processed data for LDA and BERTopic modeling

In [2]:
# Imports
import pandas as pd
import numpy as np
import re
import string
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

# NLP libraries
import nltk
import spacy
from nltk.corpus import stopwords
from sklearn.feature_extraction.text import CountVectorizer

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Settings
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 100)
plt.style.use('seaborn-v0_8-whitegrid')

%matplotlib inline

# Enable tqdm for pandas
tqdm.pandas()

print("Imports successful!")

AttributeError: module 'matplotlib' has no attribute 'get_data_path'

## 1. Download Required NLTK Data

In [ ]:
# Download NLTK stopwords
nltk.download('stopwords', quiet=True)
nltk.download('punkt', quiet=True)
nltk.download('wordnet', quiet=True)

print("NLTK data downloaded successfully!")

## 2. Load spaCy Model

In [ ]:
# Load spaCy English model (disable unused components for speed)
nlp = spacy.load("en_core_web_sm", disable=["parser", "ner"])
print(f"spaCy model loaded: {nlp.meta['name']} v{nlp.meta['version']}")

## 3. Load and Explore Dataset

In [ ]:
# Load data
df = pd.read_csv('data/Hotel_Reviews.csv')

print(f"Dataset shape: {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"\nColumns: {', '.join(df.columns.tolist())}")
df.head()

In [ ]:
# Check for placeholder reviews
no_negative = (df['Negative_Review'].str.strip().str.lower() == 'no negative').sum()
no_positive = (df['Positive_Review'].str.strip().str.lower() == 'no positive').sum()

print(f"Placeholder 'No Negative' reviews: {no_negative:,} ({no_negative/len(df)*100:.1f}%)")
print(f"Placeholder 'No Positive' reviews: {no_positive:,} ({no_positive/len(df)*100:.1f}%)")
print(f"\nTotal reviews: {len(df):,}")
print(f"Real negative reviews: {len(df) - no_negative:,}")
print(f"Real positive reviews: {len(df) - no_positive:,}")

## 4. Define Preprocessing Functions

In [ ]:
# Define domain-specific stopwords
domain_stopwords = {
    'hotel', 'room', 'stay', 'stayed', 'night', 'nights', 
    'booking', 'booked', 'reservation'
}

# Get English stopwords but keep negations
english_stops = set(stopwords.words('english'))
negation_words = {'no', 'not', 'never', 'neither', 'nobody', 'nothing', 'nowhere', 'none'}
english_stops = english_stops - negation_words

# Combine stopwords
all_stopwords = english_stops.union(domain_stopwords)

print(f"Total stopwords: {len(all_stopwords)}")
print(f"Sample stopwords: {list(all_stopwords)[:20]}")

In [ ]:
def clean_text_basic(text):
    """
    Basic text cleaning (for BERTopic - keep original context).
    
    Args:
        text: Raw review text
        
    Returns:
        Cleaned text string
    """
    if not isinstance(text, str):
        return ""
    
    # Lowercase
    text = text.lower()
    
    # Remove URLs
    text = re.sub(r'http\S+|www\S+', '', text)
    
    # Remove email addresses
    text = re.sub(r'\S+@\S+', '', text)
    
    # Remove extra whitespace
    text = re.sub(r'\s+', ' ', text).strip()
    
    return text


def clean_text_advanced(text):
    """
    Advanced text cleaning with lemmatization (for LDA).
    
    Args:
        text: Raw review text
        
    Returns:
        Lemmatized text string
    """
    if not isinstance(text, str):
        return ""
    
    # Basic cleaning
    text = clean_text_basic(text)
    
    # Process with spaCy
    doc = nlp(text)
    
    # Lemmatize and filter
    tokens = []
    for token in doc:
        # Skip if:
        # - is stop word (using our custom list)
        # - is punctuation
        # - is too short (< 3 characters)
        # - is just numbers
        if (token.lemma_.lower() not in all_stopwords and 
            not token.is_punct and 
            len(token.lemma_) >= 3 and
            not token.is_digit):
            tokens.append(token.lemma_.lower())
    
    return ' '.join(tokens)


def get_word_count(text):
    """Count words in text."""
    if not isinstance(text, str):
        return 0
    return len(text.split())


print("Preprocessing functions defined!")

# Test the functions
sample_text = "The rooms were too small and the breakfast was terrible. I didn't like it at all!"
print(f"\nOriginal: {sample_text}")
print(f"Basic clean: {clean_text_basic(sample_text)}")
print(f"Advanced clean: {clean_text_advanced(sample_text)}")

## 5. Process Negative Reviews

In [ ]:
# Filter out placeholder negative reviews
df_neg = df[df['Negative_Review'].str.strip().str.lower() != 'no negative'].copy()

print(f"Negative reviews: {len(df_neg):,} (filtered from {len(df):,})")

# Apply cleaning
print("\nApplying basic cleaning...")
df_neg['text_original'] = df_neg['Negative_Review'].progress_apply(clean_text_basic)

print("Applying advanced cleaning (lemmatization)...")
df_neg['text_cleaned'] = df_neg['Negative_Review'].progress_apply(clean_text_advanced)

# Calculate word counts
df_neg['word_count_original'] = df_neg['text_original'].apply(get_word_count)
df_neg['word_count_cleaned'] = df_neg['text_cleaned'].apply(get_word_count)

print("\nNegative reviews processing complete!")
df_neg.head()

In [ ]:
# Filter by word count (remove very short and very long reviews)
MIN_WORDS = 5
MAX_WORDS = 400

df_neg_filtered = df_neg[
    (df_neg['word_count_cleaned'] >= MIN_WORDS) & 
    (df_neg['word_count_cleaned'] <= MAX_WORDS)
].copy()

print(f"Filtered negative reviews: {len(df_neg_filtered):,}")
print(f"Removed: {len(df_neg) - len(df_neg_filtered):,} reviews")
print(f"\nWord count statistics (cleaned text):")
print(df_neg_filtered['word_count_cleaned'].describe())

In [ ]:
# Visualize word count distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax1 = axes[0]
df_neg_filtered['word_count_original'].hist(bins=50, ax=ax1, color='salmon', edgecolor='white')
ax1.set_xlabel('Word Count')
ax1.set_ylabel('Frequency')
ax1.set_title('Negative Reviews - Original Text Word Count')
ax1.axvline(df_neg_filtered['word_count_original'].mean(), color='red', linestyle='--', 
            label=f'Mean: {df_neg_filtered["word_count_original"].mean():.1f}')
ax1.legend()

ax2 = axes[1]
df_neg_filtered['word_count_cleaned'].hist(bins=50, ax=ax2, color='lightcoral', edgecolor='white')
ax2.set_xlabel('Word Count')
ax2.set_ylabel('Frequency')
ax2.set_title('Negative Reviews - Cleaned Text Word Count')
ax2.axvline(df_neg_filtered['word_count_cleaned'].mean(), color='red', linestyle='--',
            label=f'Mean: {df_neg_filtered["word_count_cleaned"].mean():.1f}')
ax2.legend()

plt.tight_layout()
plt.savefig('outputs/visualizations/negative_wordcount_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Process Positive Reviews

In [ ]:
# Filter out placeholder positive reviews
df_pos = df[df['Positive_Review'].str.strip().str.lower() != 'no positive'].copy()

print(f"Positive reviews: {len(df_pos):,} (filtered from {len(df):,})")

# Apply cleaning
print("\nApplying basic cleaning...")
df_pos['text_original'] = df_pos['Positive_Review'].progress_apply(clean_text_basic)

print("Applying advanced cleaning (lemmatization)...")
df_pos['text_cleaned'] = df_pos['Positive_Review'].progress_apply(clean_text_advanced)

# Calculate word counts
df_pos['word_count_original'] = df_pos['text_original'].apply(get_word_count)
df_pos['word_count_cleaned'] = df_pos['text_cleaned'].apply(get_word_count)

print("\nPositive reviews processing complete!")
df_pos.head()

In [ ]:
# Filter by word count
df_pos_filtered = df_pos[
    (df_pos['word_count_cleaned'] >= MIN_WORDS) & 
    (df_pos['word_count_cleaned'] <= MAX_WORDS)
].copy()

print(f"Filtered positive reviews: {len(df_pos_filtered):,}")
print(f"Removed: {len(df_pos) - len(df_pos_filtered):,} reviews")
print(f"\nWord count statistics (cleaned text):")
print(df_pos_filtered['word_count_cleaned'].describe())

In [ ]:
# Visualize word count distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax1 = axes[0]
df_pos_filtered['word_count_original'].hist(bins=50, ax=ax1, color='lightgreen', edgecolor='white')
ax1.set_xlabel('Word Count')
ax1.set_ylabel('Frequency')
ax1.set_title('Positive Reviews - Original Text Word Count')
ax1.axvline(df_pos_filtered['word_count_original'].mean(), color='green', linestyle='--',
            label=f'Mean: {df_pos_filtered["word_count_original"].mean():.1f}')
ax1.legend()

ax2 = axes[1]
df_pos_filtered['word_count_cleaned'].hist(bins=50, ax=ax2, color='mediumseagreen', edgecolor='white')
ax2.set_xlabel('Word Count')
ax2.set_ylabel('Frequency')
ax2.set_title('Positive Reviews - Cleaned Text Word Count')
ax2.axvline(df_pos_filtered['word_count_cleaned'].mean(), color='green', linestyle='--',
            label=f'Mean: {df_pos_filtered["word_count_cleaned"].mean():.1f}')
ax2.legend()

plt.tight_layout()
plt.savefig('outputs/visualizations/positive_wordcount_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Create Final Datasets

In [ ]:
# Select relevant columns for export
columns_to_keep = [
    'Hotel_Name',
    'Reviewer_Nationality', 
    'Reviewer_Score',
    'Review_Date',
    'Average_Score',
    'text_original',  # For BERTopic
    'text_cleaned',   # For LDA
    'word_count_original',
    'word_count_cleaned'
]

# Create final datasets
negative_final = df_neg_filtered[columns_to_keep].copy()
negative_final['sentiment'] = 'negative'

positive_final = df_pos_filtered[columns_to_keep].copy()
positive_final['sentiment'] = 'positive'

print(f"Final negative reviews: {len(negative_final):,}")
print(f"Final positive reviews: {len(positive_final):,}")
print(f"\nTotal reviews for topic modeling: {len(negative_final) + len(positive_final):,}")

## 8. Create Sample Datasets (for testing)

In [ ]:
# Create 50K sample for fast testing
SAMPLE_SIZE = 50000

negative_sample = negative_final.sample(n=min(SAMPLE_SIZE, len(negative_final)), random_state=42)
positive_sample = positive_final.sample(n=min(SAMPLE_SIZE, len(positive_final)), random_state=42)

print(f"Sample negative reviews: {len(negative_sample):,}")
print(f"Sample positive reviews: {len(positive_sample):,}")

## 9. Export Processed Data

In [ ]:
# Export full datasets
print("Saving full datasets...")
negative_final.to_parquet('data/negative_reviews_processed.parquet', index=False)
positive_final.to_parquet('data/positive_reviews_processed.parquet', index=False)

# Export sample datasets
print("Saving sample datasets...")
negative_sample.to_parquet('data/negative_reviews_sample.parquet', index=False)
positive_sample.to_parquet('data/positive_reviews_sample.parquet', index=False)

print("\n✅ All datasets saved successfully!")
print("\nFiles created:")
print("  - data/negative_reviews_processed.parquet (full dataset)")
print("  - data/positive_reviews_processed.parquet (full dataset)")
print("  - data/negative_reviews_sample.parquet (50K sample)")
print("  - data/positive_reviews_sample.parquet (50K sample)")

## 10. Summary Statistics

In [ ]:
# Create summary dataframe
summary = pd.DataFrame({
    'Dataset': ['Negative Reviews', 'Positive Reviews', 'Total'],
    'Count': [
        len(negative_final),
        len(positive_final),
        len(negative_final) + len(positive_final)
    ],
    'Avg Words (Original)': [
        negative_final['word_count_original'].mean(),
        positive_final['word_count_original'].mean(),
        pd.concat([negative_final, positive_final])['word_count_original'].mean()
    ],
    'Avg Words (Cleaned)': [
        negative_final['word_count_cleaned'].mean(),
        positive_final['word_count_cleaned'].mean(),
        pd.concat([negative_final, positive_final])['word_count_cleaned'].mean()
    ],
    'Avg Score': [
        negative_final['Reviewer_Score'].mean(),
        positive_final['Reviewer_Score'].mean(),
        pd.concat([negative_final, positive_final])['Reviewer_Score'].mean()
    ]
})

print("\n" + "="*80)
print("PREPROCESSING SUMMARY")
print("="*80)
print(summary.to_string(index=False))
print("="*80)

print("\n✅ Text preprocessing complete!")
print("\nNext steps:")
print("  1. Run 03_topic_modeling_lda.ipynb for LDA topic modeling")
print("  2. Run 04_topic_modeling_bertopic.ipynb for BERTopic modeling")